# Punto 2 — ID3 aplicado a una base de datos de interés profesional/investigativo
### Caso de estudio: Nivel de riesgo hídrico en subcuencas de la Orinoquia colombiana

Este ejercicio conecta el algoritmo ID3 con la línea de investigación doctoral (evaluación espacial
del riesgo climático y seguridad hídrica en la cuenca del Orinoco, bajo el marco AR6 del IPCC:
**Riesgo = f(Amenaza, Exposición, Vulnerabilidad)**).

**Fuentes de datos reales de referencia** (para un caso de estudio con datos primarios, no
sintéticos, se recomienda acudir a):
- IDEAM — Instituto de Hidrología, Meteorología y Estudios Ambientales (datos hidroclimáticos,
  http://dhime.ideam.gov.co).
- SIAC — Sistema de Información Ambiental de Colombia (uso del suelo, coberturas).
- WRI Aqueduct Water Risk Atlas (indicadores de estrés hídrico a escala de cuenca).
- DANE / proyecciones poblacionales municipales (exposición poblacional).

Dado que descargar y homologar estas fuentes excede el alcance de este ejercicio de clase, se
construye aquí una **tabla de patrones representativa** (pattern table), discretizada de la misma
forma en que se trabajaría con los indicadores reales — que es exactamente el insumo que
requiere ID3 (variables categóricas/discretas).


## 1. Modelo conceptual

**Unidad de análisis:** subcuenca / municipio de la Orinoquia.

**Entidades y relación (marco de riesgo climático IPCC-AR6):**

```
            AMENAZA HIDROCLIMÁTICA        EXPOSICIÓN POBLACIONAL
            (variabilidad de lluvias,     (población e infraestructura
             sequías/inundaciones)         en zonas afectables)
                       \                        /
                        \                      /
                         v                    v
                     +-------------------------------+
                     |   RIESGO HÍDRICO (variable     |
                     |   objetivo / clase a predecir) |
                     +-------------------------------+
                         ^                    ^
                        /                      \
            CAPACIDAD ADAPTATIVA          USO DEL SUELO
            (inversa de la vulnerabilidad  (modula la vulnerabilidad:
             socioeconómica/institucional)  agrícola, ganadero, urbano, natural)
```

**Variables de decisión (4):**

| Variable | Rol IPCC-AR6 | Valores |
|---|---|---|
| `Amenaza_hidroclimatica` | Hazard | Alta, Media, Baja |
| `Exposicion_poblacional` | Exposure | Alta, Media, Baja |
| `Capacidad_adaptativa` | Componente (inverso) de Vulnerability | Alta, Media, Baja |
| `Uso_del_suelo` | Modulador de Vulnerability | Agricola, Ganadero, Urbano, Natural |

**Variable de clase:** `Nivel_riesgo_hidrico` → Alto / Medio / Bajo


## 2. Tabla de patrones (pattern table)

Cada fila representa una unidad espacial hipotética (subcuenca) discretizada según los umbrales que
normalmente se usarían con datos reales de IDEAM/WRI Aqueduct (p. ej. terciles de precipitación,
déficit hídrico, densidad poblacional, índice de capacidad adaptativa municipal). La regla de
generación sigue la lógica del marco AR6: el riesgo tiende a ser alto cuando amenaza y exposición son
altas y la capacidad adaptativa es baja; el uso del suelo agrícola/ganadero incrementa la
vulnerabilidad frente al uso natural o urbano bien planificado.


In [2]:
import pandas as pd
import math
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

datos = [
    ("Alta",  "Alta",  "Baja",  "Ganadero", "Alto"),
    ("Alta",  "Alta",  "Media", "Agricola", "Alto"),
    ("Alta",  "Media", "Baja",  "Agricola", "Alto"),
    ("Alta",  "Baja",  "Baja",  "Natural",  "Medio"),
    ("Alta",  "Alta",  "Alta",  "Urbano",   "Medio"),
    ("Alta",  "Media", "Alta",  "Urbano",   "Medio"),
    ("Alta",  "Baja",  "Media", "Natural",  "Bajo"),
    ("Media", "Alta",  "Baja",  "Ganadero", "Alto"),
    ("Media", "Alta",  "Media", "Agricola", "Medio"),
    ("Media", "Media", "Baja",  "Agricola", "Medio"),
    ("Media", "Media", "Media", "Ganadero", "Medio"),
    ("Media", "Baja",  "Alta",  "Natural",  "Bajo"),
    ("Media", "Baja",  "Media", "Urbano",   "Bajo"),
    ("Media", "Alta",  "Alta",  "Urbano",   "Medio"),
    ("Baja",  "Alta",  "Baja",  "Agricola", "Medio"),
    ("Baja",  "Media", "Media", "Ganadero", "Bajo"),
    ("Baja",  "Baja",  "Baja",  "Natural",  "Bajo"),
    ("Baja",  "Baja",  "Alta",  "Urbano",   "Bajo"),
    ("Baja",  "Media", "Alta",  "Natural",  "Bajo"),
    ("Baja",  "Alta",  "Alta",  "Urbano",   "Bajo"),
    ("Alta",  "Media", "Baja",  "Ganadero", "Alto"),
    ("Media", "Alta",  "Baja",  "Agricola", "Alto"),
]

columnas = ["Amenaza_hidroclimatica", "Exposicion_poblacional", "Capacidad_adaptativa",
            "Uso_del_suelo", "Nivel_riesgo_hidrico"]
df2 = pd.DataFrame(datos, columns=columnas)
df2


,Amenaza_hidroclimatica,Exposicion_poblacional,Capacidad_adaptativa,Uso_del_suelo,Nivel_riesgo_hidrico
0,Alta,Alta,Baja,Ganadero,Alto
1,Alta,Alta,Media,Agricola,Alto
2,Alta,Media,Baja,Agricola,Alto
3,Alta,Baja,Baja,Natural,Medio
4,Alta,Alta,Alta,Urbano,Medio
5,Alta,Media,Alta,Urbano,Medio
6,Alta,Baja,Media,Natural,Bajo
7,Media,Alta,Baja,Ganadero,Alto
8,Media,Alta,Media,Agricola,Medio
9,Media,Media,Baja,Agricola,Medio


## 3. Reutilización del motor ID3 (mismas funciones del Punto 1)

In [3]:
def entropia(serie_clase):
    conteos = Counter(serie_clase)
    total = len(serie_clase)
    ent = 0.0
    for clase, n in conteos.items():
        p = n / total
        ent -= p * math.log2(p)
    return ent


def ganancia_informacion(df, atributo, clase, mostrar=True):
    ent_total = entropia(df[clase])
    ent_ponderada = 0.0
    detalle = []
    for valor in sorted(df[atributo].unique()):
        subconjunto = df[df[atributo] == valor]
        p = len(subconjunto) / len(df)
        e = entropia(subconjunto[clase])
        ent_ponderada += p * e
        detalle.append((valor, len(subconjunto), round(e, 4)))
    ganancia = ent_total - ent_ponderada
    if mostrar:
        print(f"  Entropia(S) = {ent_total:.4f}")
        for valor, n, e in detalle:
            print(f"    -> {atributo}={valor:<10} n={n:<3} Entropia={e:.4f}")
        print(f"  Entropia ponderada tras dividir por '{atributo}' = {ent_ponderada:.4f}")
        print(f"  Ganancia({atributo}) = {ent_total:.4f} - {ent_ponderada:.4f} = {ganancia:.4f}\n")
    return ganancia


def id3(df, atributos, clase, nivel=0, traza=True):
    indent = "  " * nivel
    clases_presentes = df[clase].unique()

    if len(clases_presentes) == 1:
        if traza:
            print(f"{indent}Hoja -> '{clases_presentes[0]}' (nodo puro, Entropia=0)")
        return clases_presentes[0]

    if not atributos:
        mayoria = df[clase].mode()[0]
        if traza:
            print(f"{indent}Hoja (sin atributos restantes) -> clase mayoritaria: '{mayoria}'")
        return mayoria

    if traza:
        print(f"{indent}Nodo con {len(df)} ejemplos. Evaluando atributos {atributos}:")
    ganancias = {}
    for atributo in atributos:
        ganancias[atributo] = ganancia_informacion(df, atributo, clase, mostrar=False)
        if traza:
            print(f"{indent}  Ganancia({atributo}) = {ganancias[atributo]:.4f}")

    mejor_atributo = max(ganancias, key=ganancias.get)
    if traza:
        print(f"{indent}>>> Se divide por '{mejor_atributo}'\n")

    arbol = {mejor_atributo: {}}
    atributos_restantes = [a for a in atributos if a != mejor_atributo]

    for valor in sorted(df[mejor_atributo].unique()):
        subconjunto = df[df[mejor_atributo] == valor]
        if traza:
            print(f"{indent}Rama {mejor_atributo} = {valor} ({len(subconjunto)} ejemplos)")
        if len(subconjunto) == 0:
            arbol[mejor_atributo][valor] = df[clase].mode()[0]
        else:
            arbol[mejor_atributo][valor] = id3(
                subconjunto, atributos_restantes, clase, nivel + 1, traza
            )
    return arbol


def imprimir_reglas(arbol, condiciones=None):
    if condiciones is None:
        condiciones = []
    if not isinstance(arbol, dict):
        antecedente = " Y ".join(condiciones)
        print(f"SI {antecedente}\n   ENTONCES Nivel_riesgo_hidrico = {arbol}\n")
        return
    atributo = next(iter(arbol))
    for valor, subarbol in arbol[atributo].items():
        imprimir_reglas(subarbol, condiciones + [f"{atributo} = {valor}"])


def predecir(arbol, ejemplo):
    if not isinstance(arbol, dict):
        return arbol
    atributo = next(iter(arbol))
    valor = ejemplo.get(atributo)
    subarbol = arbol[atributo].get(valor)
    if subarbol is None:
        return "Desconocido (combinacion no vista en el entrenamiento)"
    return predecir(subarbol, ejemplo)


## 4. Entropía inicial y ganancias de información en la raíz

In [4]:
atributos2 = ["Amenaza_hidroclimatica", "Exposicion_poblacional",
              "Capacidad_adaptativa", "Uso_del_suelo"]

print("Distribucion de la clase 'Nivel_riesgo_hidrico':")
print(df2["Nivel_riesgo_hidrico"].value_counts(), "\n")
print(f"Entropia(S) del conjunto completo = {entropia(df2['Nivel_riesgo_hidrico']):.4f} bits\n")

for atributo in atributos2:
    print(f"--- Ganancia de informacion para '{atributo}' ---")
    ganancia_informacion(df2, atributo, "Nivel_riesgo_hidrico")


Distribucion de la clase 'Nivel_riesgo_hidrico':
Nivel_riesgo_hidrico
Medio    8
Bajo     8
Alto     6
Name: count, dtype: int64 

Entropia(S) del conjunto completo = 1.5726 bits

--- Ganancia de informacion para 'Amenaza_hidroclimatica' ---
  Entropia(S) = 1.5726
    -> Amenaza_hidroclimatica=Alta       n=8   Entropia=1.4056
    -> Amenaza_hidroclimatica=Baja       n=6   Entropia=0.6500
    -> Amenaza_hidroclimatica=Media      n=8   Entropia=1.5000
  Entropia ponderada tras dividir por 'Amenaza_hidroclimatica' = 1.2339
  Ganancia(Amenaza_hidroclimatica) = 1.5726 - 1.2339 = 0.3387

--- Ganancia de informacion para 'Exposicion_poblacional' ---
  Entropia(S) = 1.5726
    -> Exposicion_poblacional=Alta       n=9   Entropia=1.3921
    -> Exposicion_poblacional=Baja       n=6   Entropia=0.6500
    -> Exposicion_poblacional=Media      n=7   Entropia=1.5567
  Entropia ponderada tras dividir por 'Exposicion_poblacional' = 1.2421
  Ganancia(Exposicion_poblacional) = 1.5726 - 1.2421 = 0.3305

--

## 5. Construcción del árbol ID3 completo

In [5]:
arbol_riesgo = id3(df2, atributos2, "Nivel_riesgo_hidrico")


Nodo con 22 ejemplos. Evaluando atributos ['Amenaza_hidroclimatica', 'Exposicion_poblacional', 'Capacidad_adaptativa', 'Uso_del_suelo']:
  Ganancia(Amenaza_hidroclimatica) = 0.3387
  Ganancia(Exposicion_poblacional) = 0.3305
  Ganancia(Capacidad_adaptativa) = 0.3082
  Ganancia(Uso_del_suelo) = 0.5515
>>> Se divide por 'Uso_del_suelo'

Rama Uso_del_suelo = Agricola (6 ejemplos)
  Nodo con 6 ejemplos. Evaluando atributos ['Amenaza_hidroclimatica', 'Exposicion_poblacional', 'Capacidad_adaptativa']:
    Ganancia(Amenaza_hidroclimatica) = 0.5409
    Ganancia(Exposicion_poblacional) = 0.0000
    Ganancia(Capacidad_adaptativa) = 0.0000
  >>> Se divide por 'Amenaza_hidroclimatica'

  Rama Amenaza_hidroclimatica = Alta (2 ejemplos)
    Hoja -> 'Alto' (nodo puro, Entropia=0)
  Rama Amenaza_hidroclimatica = Baja (1 ejemplos)
    Hoja -> 'Medio' (nodo puro, Entropia=0)
  Rama Amenaza_hidroclimatica = Media (3 ejemplos)
    Nodo con 3 ejemplos. Evaluando atributos ['Exposicion_poblacional', 'Capaci

In [6]:
import pprint
print("Arbol como estructura de diccionarios:\n")
pprint.pprint(arbol_riesgo)
print("\nReglas de decision inducidas:\n")
imprimir_reglas(arbol_riesgo)


Arbol como estructura de diccionarios:

{'Uso_del_suelo': {'Agricola': {'Amenaza_hidroclimatica': {'Alta': 'Alto',
                                                           'Baja': 'Medio',
                                                           'Media': {'Exposicion_poblacional': {'Alta': {'Capacidad_adaptativa': {'Baja': 'Alto',
                                                                                                                                  'Media': 'Medio'}},
                                                                                                'Media': 'Medio'}}}},
                   'Ganadero': {'Amenaza_hidroclimatica': {'Alta': 'Alto',
                                                           'Baja': 'Bajo',
                                                           'Media': {'Exposicion_poblacional': {'Alta': 'Alto',
                                                                                                'Media': 'Medio'}}}},
               

## 6. Clasificación de nuevas subcuencas hipotéticas

In [7]:
nuevas_subcuencas = [
    {"Amenaza_hidroclimatica": "Alta", "Exposicion_poblacional": "Alta",
     "Capacidad_adaptativa": "Baja", "Uso_del_suelo": "Agricola"},
    {"Amenaza_hidroclimatica": "Baja", "Exposicion_poblacional": "Baja",
     "Capacidad_adaptativa": "Alta", "Uso_del_suelo": "Natural"},
    {"Amenaza_hidroclimatica": "Media", "Exposicion_poblacional": "Media",
     "Capacidad_adaptativa": "Media", "Uso_del_suelo": "Urbano"},
]

for caso in nuevas_subcuencas:
    print(caso, "->", predecir(arbol_riesgo, caso))


{'Amenaza_hidroclimatica': 'Alta', 'Exposicion_poblacional': 'Alta', 'Capacidad_adaptativa': 'Baja', 'Uso_del_suelo': 'Agricola'} -> Alto
{'Amenaza_hidroclimatica': 'Baja', 'Exposicion_poblacional': 'Baja', 'Capacidad_adaptativa': 'Alta', 'Uso_del_suelo': 'Natural'} -> Bajo
{'Amenaza_hidroclimatica': 'Media', 'Exposicion_poblacional': 'Media', 'Capacidad_adaptativa': 'Media', 'Uso_del_suelo': 'Urbano'} -> Desconocido (combinacion no vista en el entrenamiento)


## 7. Exportación a formato ARFF (para verificación cruzada en WEKA)

El enunciado recomienda emplear la plataforma WEKA para este punto. Aquí se exporta la tabla de
patrones a formato `.arff` para que, si se desea, el mismo dataset se pueda cargar directamente en
WEKA (Explorer → `classifiers/trees/Id3`, nota: el `Id3` de WEKA solo acepta atributos nominales,
que es justamente el tipo de dato usado aquí) y comparar el árbol obtenido contra el generado en
Python.


In [8]:
def exportar_arff(df, nombre_relacion, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(f"@RELATION {nombre_relacion}\n\n")
        for columna in df.columns:
            valores = ",".join(sorted(df[columna].unique()))
            f.write(f"@ATTRIBUTE {columna} {{{valores}}}\n")
        f.write("\n@DATA\n")
        for _, fila in df.iterrows():
            f.write(",".join(fila.astype(str)) + "\n")

exportar_arff(df2, "riesgo_hidrico_orinoquia", "dataset_riesgo_hidrico.arff")
print("Archivo dataset_riesgo_hidrico.arff generado.")


Archivo dataset_riesgo_hidrico.arff generado.


## 8. Comentario de resultados

- **Atributo raíz:** contra lo que podría esperarse a priori, el atributo con **mayor ganancia de
  información en la raíz fue `Uso_del_suelo` (Ganancia = 0.5515)**, por encima de
  `Amenaza_hidroclimatica` (0.3387), `Exposicion_poblacional` (0.3305) y `Capacidad_adaptativa`
  (0.3082). Esto ocurre porque, en la tabla de patrones construida, la clase `Uso_del_suelo=Natural`
  concentra casi exclusivamente riesgo `Bajo` (entropía 0.72), mientras que `Agricola` y `Urbano`
  quedan con entropía máxima (1.0): estadísticamente esa partición reduce más la incertidumbre
  global que cualquiera de las otras tres variables por sí solas.
- **Lectura conceptual:** este resultado es coherente con el papel del uso del suelo como
  **modulador de la vulnerabilidad** dentro del marco AR6 — en el dataset construido termina siendo
  la variable más discriminante porque combina, indirectamente, información de exposición y de
  capacidad de respuesta del territorio. Sin embargo, es una **advertencia metodológica
  importante**: con una tabla de patrones pequeña (22 registros) y construida manualmente, ID3 puede
  sobreponderar variables que "casualmente" separan bien la muestra sin que ello refleje
  necesariamente el peso real del factor en campo. Con datos reales de mayor tamaño (IDEAM/WRI
  Aqueduct) es esperable que `Amenaza_hidroclimatica` y `Exposicion_poblacional` recuperen
  protagonismo.
- **Estructura del árbol:** dentro de cada rama de uso del suelo, el algoritmo continúa dividiendo
  por `Amenaza_hidroclimatica` y, en los casos aún mixtos, por `Exposicion_poblacional` o
  `Capacidad_adaptativa` — reproduciendo la lógica esperada de que Amenaza=Alta combinada con baja
  capacidad adaptativa tiende a producir riesgo Alto, y Amenaza=Baja tiende a Bajo, salvo que el
  uso del suelo ya haya decidido la clase antes.
- **Limitación observada en la predicción:** el caso hipotético
  `Amenaza=Media, Exposición=Media, Capacidad_adaptativa=Media, Uso_del_suelo=Urbano` fue
  clasificado como **"Desconocido"**, porque esa combinación exacta no apareció en el entrenamiento.
  Esto ilustra una debilidad conocida de ID3: no generaliza más allá de las combinaciones observadas
  ni maneja incertidumbre por defecto (a diferencia de C4.5/J48, que sí puede asignar la clase
  mayoritaria del nodo más cercano). Es un argumento concreto para contrastar el resultado con J48 en
  WEKA sobre el mismo archivo `.arff`.
- **Utilidad para la tesis doctoral:** el flujo completo (modelo conceptual → tabla de patrones →
  ID3 → reglas SI-ENTONCES) es directamente extrapolable a datos reales discretizados en terciles o
  quintiles, y genera reglas interpretables — más fáciles de comunicar a tomadores de decisión que un
  modelo de caja negra — lo cual es valioso en evaluación multicriterio de riesgo climático y en la
  comunicación de resultados de la investigación en la cuenca del Orinoco.
